# Car prices — held-out testing and model comparison
This notebook evaluates the models saved by notebook 04 on cars held out from training and validation. Known prices and predictions are matched by source RowId. Saved data and feature checks keep the evaluation tied to the same training run.

The comparison covers overall error, residuals, percentage differences and price bands, including cars below 50k. Models are scored from their own predictions, and the lowest held-out RMSE is highlighted in the report.


## Setup and matching cleaning/feature definitions


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Locate data from the notebook folder, project folder or repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "car_price_prediction")
     if (candidate / "car_prices_dataset" / "train" / "car_price.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or car_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "car_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "car_price.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "Price"
ID_COLUMN = "RowId"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
def clean_features(X, reference_year):
    """Apply row-wise validity rules; fitted imputers handle remaining gaps."""
    data = X.drop(columns=[ID_COLUMN, TARGET, "Model"], errors="ignore").copy()
    for column in ("Mileage", "EngineV", "Year"):
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data.loc[~np.isfinite(data[column]), column] = np.nan
    data.loc[data["Mileage"] < 0, "Mileage"] = np.nan
    # Treat engine volumes above 6.5 L as unknown for this dataset.
    # Treat them as unknown; retain the car instead of deleting its record.
    data.loc[(data["EngineV"] <= 0) | (data["EngineV"] > 6.5), "EngineV"] = np.nan
    data.loc[(data["Year"] < 1886) | (data["Year"] > reference_year), "Year"] = np.nan
    for column in data.select_dtypes(exclude="number").columns:
        data[column] = data[column].map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
        data[column] = data[column].replace("", np.nan)
    return data

def engineer_features(X, reference_year):
    """Keep domain features deterministic and independent of prices."""
    data = clean_features(X, reference_year)
    data["AgeAtSnapshot"] = reference_year - data["Year"]
    data["LogMileage"] = np.log1p(data["Mileage"])
    # Mileage units are preserved from the source; no unit conversion is assumed.
    data["MileagePerYear"] = data["Mileage"] / (data["AgeAtSnapshot"] + 1)
    data["EngineVolumeMissing"] = data["EngineV"].isna().astype(int)
    # Age and Year encode the same linear information; keep one representation.
    return data.drop(columns="Year")


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metrics(actual, predicted):
    predicted = np.maximum(0, np.asarray(predicted))
    if not np.isfinite(predicted).all():
        raise ValueError("Nonfinite model predictions")
    actual = np.asarray(actual)
    percentage_error = np.abs(actual - predicted) / actual * 100
    return {"mape_percent": float(percentage_error.mean()),
            "within_50_percent": float(np.mean(percentage_error <= 50) * 100),
            "mae": float(mean_absolute_error(actual, predicted)),
            "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
            "rmsle": float(np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(predicted)))),
            "r2": float(r2_score(actual, predicted))}


## Load and verify the saved training run


In [4]:
import hashlib
import inspect
import json
import joblib

required = ["run_metadata.json", "candidates.joblib", "model_comparison.csv"]
missing = [name for name in required if not (ARTIFACTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Run 04_training.ipynb first. Missing artifacts: {missing}")
metadata = json.loads((ARTIFACTS_DIR / "run_metadata.json").read_text())
if metadata.get("format_version") != 1 or metadata.get("project") != "car_price_prediction":
    raise ValueError("Unsupported saved run. Rerun 04_training.ipynb.")
for relative, expected in metadata["file_sha256"].items():
    path = PROJECT_ROOT / relative
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Data or training artifact changed: {relative}. Rerun 04_training.ipynb.")
signature = hashlib.sha256((inspect.getsource(clean_features) + inspect.getsource(engineer_features)).encode()).hexdigest()
if signature != metadata["feature_signature"]:
    raise ValueError("Feature definitions differ from training. Keep the feature cells in 03, 04 and 05 in sync and rerun training.")
config = metadata["config"]
display(pd.Series(config, name="saved_training_configuration").to_frame())
X_raw = pd.read_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv")
ground = pd.read_csv(GROUND_TRUTH_DIR / "ground_truth.csv")
if not X_raw[ID_COLUMN].equals(ground[ID_COLUMN]) or ground[ID_COLUMN].duplicated().any():
    raise ValueError("Ground-truth identifiers do not match unique prediction rows. Rerun training.")
X = engineer_features(X_raw, config["reference_year"])
candidates = joblib.load(ARTIFACTS_DIR / "candidates.joblib")
if list(candidates) != metadata["models"]:
    raise ValueError("Saved models do not match run metadata. Rerun training.")
training_comparison = pd.read_csv(ARTIFACTS_DIR / "model_comparison.csv")
if set(training_comparison.model) != set(candidates):
    raise ValueError("Training scores do not match saved models. Rerun training.")
print("Loaded candidates:", ", ".join(candidates))


,saved_training_configuration
random_state,42
ground_truth_fraction,0.2
validation_fraction,0.2
cv_folds,5
models,None
tune,True
tune_top_n,2
overfit_ratio,1.5
underperformance_ratio,1.25
reference_year,2016


Loaded candidates: DummyMedian, LinearRegression, DecisionTree, Ridge, Lasso, ElasticNet, RandomForest, GradientBoostedTreesModel, HistogramGradientBoostedTreesModel, XGBoost, DecisionTree (tuned), RandomForest (tuned)


## Reporting functions


In [5]:
from html import escape
import plotly.express as px
import plotly.graph_objects as go
from plotly.io import to_html


def _table(df):
    return df.to_html(index=False, classes="dataframe", border=0, float_format=lambda x: f"{x:,.4f}")


def write_html(path: Path, title: str, intro: str, tables=None, figures=None, notes=None):
    tables = tables or []
    figures = figures or []
    notes = notes or []
    body = [f"<h1>{escape(title)}</h1><p>{escape(intro)}</p>"]
    for heading, table in tables:
        body.append(f"<h2>{escape(heading)}</h2>{_table(table)}")
    for index, (heading, figure) in enumerate(figures):
        body.append(f"<h2>{escape(heading)}</h2>{to_html(figure, full_html=False, include_plotlyjs=True if index == 0 else False)}")
    if notes:
        body.append("<h2>Evaluation notes</h2><ul>" + "".join(f"<li>{escape(n)}</li>" for n in notes) + "</ul>")
    html = """<!doctype html><html><head><meta charset='utf-8'><title>{}</title>
    <style>body{{font-family:Arial,sans-serif;max-width:1200px;margin:2rem auto;padding:0 1rem;color:#243447}}h1{{color:#12355b}}h2{{margin-top:2rem}}.dataframe{{border-collapse:collapse;width:100%}}.dataframe th,.dataframe td{{border:1px solid #ddd;padding:7px;text-align:right}}.dataframe th{{background:#eaf2f8;text-align:center}}</style>
    </head><body>{}</body></html>""".format(escape(title), "".join(body))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")


def comparison_figure(results: pd.DataFrame):
    return px.bar(results.sort_values("cv_rmse"), x="cv_rmse", y="model", orientation="h", error_x="cv_rmse_std", title="Cross-validation RMSE on Price scale")


def stage_comparison_figure(results: pd.DataFrame, metric="rmse"):
    columns = [f"{stage}_{metric}" for stage in ("train", "validation", "test") if f"{stage}_{metric}" in results]
    flags = [name for name in ("overfitting_flag", "best_model_flag", "underperforming_flag") if name in results]
    frame = results.melt(id_vars=["model", *flags], value_vars=columns, var_name="dataset", value_name=metric)
    frame["dataset"] = frame["dataset"].str.replace(f"_{metric}", "", regex=False).map({"train": "Training", "validation": "Validation", "test": "Ground-truth testing"})
    direction = "lower is better" if metric != "r2" else "higher is better"
    figure = px.bar(frame, x="model", y=metric, color="dataset", barmode="group", hover_data=flags,
                     title=f"{metric.upper()}: training vs testing ({direction})")
    figure.update_layout(xaxis_tickangle=-35)
    return figure


def actual_predicted_figure(y_true, y_pred, title):
    frame = pd.DataFrame({"actual": y_true, "predicted": y_pred})
    fig = px.scatter(frame, x="actual", y="predicted", title=title, opacity=0.7)
    limits = [min(frame.min()), max(frame.max())]
    fig.add_trace(go.Scatter(x=limits, y=limits, mode="lines", name="perfect prediction"))
    return fig


def residual_figure(y_true, y_pred):
    frame = pd.DataFrame({"predicted": y_pred, "residual": y_true - y_pred})
    fig = px.scatter(frame, x="predicted", y="residual", title="Holdout residuals", opacity=0.7)
    fig.add_hline(y=0)
    return fig


## Evaluate all candidates and generate reports


In [6]:
records = []
predictions = pd.DataFrame({ID_COLUMN: ground[ID_COLUMN], "actual": ground[TARGET]})
for name, model in candidates.items():
    predicted = np.maximum(0, model.predict(X))
    records.append({"model": name, **metrics(ground[TARGET], predicted)})
    predictions[name] = predicted
results = pd.DataFrame(records).sort_values("rmse").reset_index(drop=True)
minimum = results.rmse.min()
results["best_model_flag"] = np.isclose(results.rmse, minimum, rtol=1e-9, atol=1e-9)
results["error_vs_best_ratio"] = results.rmse / max(minimum, 1e-9)
results["underperforming_flag"] = results.error_vs_best_ratio >= config["underperformance_ratio"]
best_names = results.loc[results.best_model_flag, "model"].tolist()
diagnostic_best = best_names[0]
results.to_csv(GROUND_TRUTH_DIR / "ground_truth_comparison.csv", index=False)
predictions.to_csv(GROUND_TRUTH_DIR / "ground_truth_predictions.csv", index=False)
combined = training_comparison.merge(
    results.rename(columns={name: f"test_{name}" for name in ("mae", "rmse", "rmsle", "r2", "mape_percent", "within_50_percent")}), on="model", validate="one_to_one")
combined = combined.sort_values("test_rmse").reset_index(drop=True)
combined.to_csv(ARTIFACTS_DIR / "training_testing_comparison.csv", index=False)
metadata["best_observed_models"] = best_names
(ARTIFACTS_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))
# Summarize relative errors and cars priced below 50k.
detail_frames = []
for name in candidates:
    detail = X_raw[[ID_COLUMN, "Brand", "Body", "Model", "Year", "Mileage"]].copy()
    detail["model"] = name
    detail["actual"] = ground[TARGET].to_numpy()
    detail["predicted"] = predictions[name].to_numpy()
    detail["residual"] = detail["actual"] - detail["predicted"]
    detail["percentage_error"] = detail["residual"].abs() / detail["actual"] * 100
    detail_frames.append(detail)
error_details = pd.concat(detail_frames, ignore_index=True)
error_details["price_band"] = pd.cut(error_details.actual, bins=[0, 5000, 50000, np.inf],
                                     labels=["Below 5k", "5k to below 50k", "50k and above"], right=False)
band_records = []
for (name, band), rows in error_details.groupby(["model", "price_band"], observed=True):
    band_records.append({"model": name, "price_band": str(band), "count": len(rows),
                         **metrics(rows.actual, rows.predicted)})
band_scores = pd.DataFrame(band_records)
error_details.to_csv(ARTIFACTS_DIR / "prediction_errors.csv", index=False)
band_scores.to_csv(ARTIFACTS_DIR / "price_band_comparison.csv", index=False)
write_html(REPORTS_DIR / "testing_evaluation.html", "Ground-truth evaluation",
           "Every training-fitted candidate is compared against reserved known prices.",
           tables=[("Ground-truth ranking and flags", results), ("Price-band errors", band_scores), ("Largest percentage errors", error_details.sort_values("percentage_error", ascending=False).head(30))],
           figures=[("Actual vs predicted: lowest-RMSE candidate", actual_predicted_figure(ground[TARGET], predictions[diagnostic_best], diagnostic_best)),
                    ("Residuals: lowest-RMSE candidate", residual_figure(ground[TARGET], predictions[diagnostic_best]))],
           notes=["Known prices are aligned by source RowId for scoring.", f"Best observed by test RMSE: {', '.join(best_names)}. The flag marks the lowest error in this comparison.",
                  f"Underperforming means test RMSE is at least {config['underperformance_ratio']} times the best observed RMSE."])
write_html(REPORTS_DIR / "model_comparison.html", "Model comparison",
           f"Best observed by ground-truth RMSE: {', '.join(best_names)}. The ranking summarizes this held-out split.",
           tables=[("Training and testing scores", combined)],
           figures=[("Dollar errors", stage_comparison_figure(combined)), ("Explained variance", stage_comparison_figure(combined, "r2"))],
           notes=["Training, validation and testing predictions use the same candidate fits; no refit on validation or ground truth.",
                  "Best-model and underperformance flags are relative to test RMSE; overfitting flags use validation/training RMSE.",
                  "The workflow ends at comparison. No candidate is selected, promoted or deployed."])
print(f"Best observed by ground-truth RMSE: {', '.join(best_names)}. No model selected.", flush=True)
comparison = results
display(comparison)
display(predictions.head(15))


Best observed by ground-truth RMSE: RandomForest (tuned), RandomForest. No model selected.


,model,mape_percent,within_50_percent,mae,rmse,rmsle,r2,best_model_flag,error_vs_best_ratio,underperforming_flag
0,RandomForest (tuned),20.270185,92.484848,3106.413558,7663.386268,0.278360,0.876338,True,1.000000,False
1,RandomForest,20.270185,92.484848,3106.413558,7663.386268,0.278360,0.876338,True,1.000000,False
2,XGBoost,19.792737,92.969697,3300.523180,8318.586218,0.269212,0.854289,False,1.085497,False
3,HistogramGradientBoostedTreesModel,20.436698,93.333333,3309.792838,8320.032461,0.274914,0.854238,False,1.085686,False
4,GradientBoostedTreesModel,20.247176,93.333333,3323.337918,8433.264747,0.278305,0.850244,False,1.100462,False
5,DecisionTree (tuned),25.962200,90.424242,4354.397466,10108.062013,0.337899,0.784856,False,1.319007,True
6,LinearRegression,24.351857,89.333333,4279.105670,10953.681023,0.319497,0.747353,False,1.429353,True
7,ElasticNet,24.346407,89.333333,4289.954702,10967.191264,0.319343,0.746729,False,1.431116,True
8,Lasso,24.335712,89.454545,4296.306767,10974.704179,0.319224,0.746382,False,1.432096,True
9,Ridge,24.446328,89.454545,4317.058844,10993.828775,0.319679,0.745498,False,1.434591,True


,RowId,actual,DummyMedian,LinearRegression,DecisionTree,Ridge,Lasso,ElasticNet,RandomForest,GradientBoostedTreesModel,HistogramGradientBoostedTreesModel,XGBoost,DecisionTree (tuned),RandomForest (tuned)
0,1690,42900.0,11700.0,41061.579284,93555.00,40827.098943,41645.159332,41354.712849,54412.800461,44702.695071,52257.338957,46226.941406,58151.677196,54412.800461
1,932,30999.0,11700.0,33355.138446,29602.95,32877.715766,33081.759476,33185.863198,30887.552676,24698.799061,32406.560193,28313.792969,31220.690197,30887.552676
2,325,8100.0,11700.0,8478.327760,5000.00,8586.042299,8477.395896,8483.123507,6974.113550,8942.314302,7151.369648,8171.285645,6707.608733,6974.113550
3,1963,11500.0,11700.0,11835.345653,8999.00,11794.514960,11807.733253,11819.841114,10437.516934,11490.823416,10417.174520,11818.514648,10157.035828,10437.516934
4,1243,5800.0,11700.0,4316.832503,6000.00,4306.317522,4321.532942,4318.634302,4428.146402,4948.453062,4637.250190,4451.004395,5095.461689,4428.146402
5,1871,4900.0,11700.0,6268.373051,8200.00,6224.106556,6260.883479,6261.363932,5651.252239,5837.974863,5805.072629,5597.162598,6931.942005,5651.252239
6,195,9200.0,11700.0,9367.428104,6500.00,9353.476838,9368.242173,9367.569108,9721.674140,9523.390443,9268.646137,9200.377930,8640.221503,9721.674140
7,1688,3800.0,11700.0,4049.817256,5300.00,4050.694312,4011.957517,4029.931753,3774.491498,3775.534964,3417.792106,3708.218262,3630.173495,3774.491498
8,3059,13200.0,11700.0,14852.342675,12999.00,14962.394468,14958.050066,14911.297079,13138.169526,14807.218123,14079.588517,13499.960938,11275.607452,13138.169526
9,324,9100.0,11700.0,10072.109594,11600.00,9945.936141,10058.107934,10056.827647,9891.066145,10312.494384,9643.086824,10020.109375,13088.327204,9891.066145


## Price bands and prediction errors
Random Forest and its tuned variant tie for the lowest held-out RMSE at 7,663, with MAE of 3,106 and R² of 0.876. Their predictions are identical in this run. XGBoost follows with RMSE of 8,319 and has slightly lower mean percentage error, 19.79% versus 20.27% for the forest.

The forest's mean percentage error is 35.39% below 5k, compared with 16.78% from 5k to below 50k. The 50 cars priced at 50k or more have RMSE of 26,918, showing how the upper price band contributes large absolute misses. The untuned decision tree's strong validation result does not carry over to this set.

Residuals are actual minus predicted, so negative values indicate overprediction. The within-50% measure counts predictions at or inside that tolerance; it is a descriptive error threshold. A useful next check is the largest errors in each price band, with any model changes evaluated through training CV.


In [7]:
display(band_scores)
display(error_details.sort_values("percentage_error", ascending=False).head(20))
below_50k = error_details.loc[error_details.actual < 50000]
display(below_50k.groupby("model").agg(
    rows=("actual", "size"),
    within_50_percent=("percentage_error", lambda values: (values <= 50).mean() * 100),
    mean_percentage_error=("percentage_error", "mean"),
))
px.bar(band_scores, x="model", y="within_50_percent", color="price_band", barmode="group", title="Predictions within 50% of actual price by price band").show()


,model,price_band,count,mape_percent,within_50_percent,mae,rmse,rmsle,r2
0,DecisionTree,Below 5k,151,45.435512,78.145695,1290.069283,2088.739132,0.471883,-3.333195
1,DecisionTree,5k to below 50k,624,23.155217,89.583333,3660.642291,7528.775875,0.333470,0.461607
2,DecisionTree,50k and above,50,24.175820,86.000000,21568.574063,41678.703493,0.407168,-0.568377
3,DecisionTree (tuned),Below 5k,151,40.472691,80.794702,1125.514577,1732.747550,0.429947,-1.982019
4,DecisionTree (tuned),5k to below 50k,624,22.441779,92.948718,3596.347584,5845.217763,0.301896,0.675472
5,DecisionTree (tuned),50k and above,50,26.075377,88.000000,23566.086325,35360.812331,0.433821,-0.128929
6,DummyMedian,Below 5k,151,311.457655,0.000000,8504.788079,8563.776016,1.394973,-71.840079
7,DummyMedian,5k to below 50k,624,43.127710,60.897436,7518.449551,11180.031161,0.596402,-0.187235
8,DummyMedian,50k and above,50,84.983653,0.000000,74902.491000,81963.224741,1.970419,-5.065411
9,ElasticNet,Below 5k,151,36.684321,81.456954,1068.339327,2160.548405,0.413388,-3.636260


,RowId,Brand,Body,Model,Year,Mileage,model,actual,predicted,residual,percentage_error,price_band
540,3948,Toyota,van,Lite Ace,1985,140,DummyMedian,1100.0,11700.000000,-10600.000000,963.636364,Below 5k
820,500,Volkswagen,other,Caddy,2000,300,DummyMedian,1200.0,11700.000000,-10500.000000,875.000000,Below 5k
638,205,Mitsubishi,hatch,Colt,1996,245,DummyMedian,1389.0,11700.000000,-10311.000000,742.332613,Below 5k
3087,3706,Mitsubishi,crossover,L 200,2007,1,Ridge,2500.0,20858.174243,-18358.174243,734.326970,Below 5k
4737,3706,Mitsubishi,crossover,L 200,2007,1,ElasticNet,2500.0,20580.090232,-18080.090232,723.203609,Below 5k
3912,3706,Mitsubishi,crossover,L 200,2007,1,Lasso,2500.0,20572.884723,-18072.884723,722.915389,Below 5k
1437,3706,Mitsubishi,crossover,L 200,2007,1,LinearRegression,2500.0,20550.241703,-18050.241703,722.009668,Below 5k
348,1100,Volkswagen,other,Jetta,1987,200,DummyMedian,1450.0,11700.000000,-10250.000000,706.896552,Below 5k
291,1479,Mitsubishi,crossover,Pajero,1996,245,DummyMedian,1450.0,11700.000000,-10250.000000,706.896552,Below 5k
662,2212,Mitsubishi,vagon,Space Star,2000,185,DummyMedian,1600.0,11700.000000,-10100.000000,631.250000,Below 5k


,rows,within_50_percent,mean_percentage_error
model,,,
DecisionTree,775,87.354839,27.496281
DecisionTree (tuned),775,90.580645,25.954898
DummyMedian,775,49.032258,95.408770
ElasticNet,775,89.677419,24.041381
GradientBoostedTreesModel,775,93.419355,20.258863
HistogramGradientBoostedTreesModel,775,93.419355,20.439434
Lasso,775,89.806452,24.023609
LinearRegression,775,89.677419,24.056987
RandomForest,775,92.645161,20.401559


In [8]:
print("Best observed by ground-truth RMSE:", ", ".join(metadata["best_observed_models"]))
display(comparison[["model", "rmse", "mae", "r2", "best_model_flag", "underperforming_flag"]])
display(combined)
stage_comparison_figure(combined).show()
stage_comparison_figure(combined, "r2").show()
for name in ["training_performance", "testing_evaluation", "model_comparison"]:
    print("Report:", REPORTS_DIR / f"{name}.html")
    display(HTML(f'<a href="../reports/{name}.html" target="_blank">{name}</a>'))


Best observed by ground-truth RMSE: RandomForest (tuned), RandomForest


,model,rmse,mae,r2,best_model_flag,underperforming_flag
0,RandomForest (tuned),7663.386268,3106.413558,0.876338,True,False
1,RandomForest,7663.386268,3106.413558,0.876338,True,False
2,XGBoost,8318.586218,3300.523180,0.854289,False,False
3,HistogramGradientBoostedTreesModel,8320.032461,3309.792838,0.854238,False,False
4,GradientBoostedTreesModel,8433.264747,3323.337918,0.850244,False,False
5,DecisionTree (tuned),10108.062013,4354.397466,0.784856,False,True
6,LinearRegression,10953.681023,4279.105670,0.747353,False,True
7,ElasticNet,10967.191264,4289.954702,0.746729,False,True
8,Lasso,10974.704179,4296.306767,0.746382,False,True
9,Ridge,10993.828775,4317.058844,0.745498,False,True


,model,cv_rmse,cv_rmse_std,train_mape_percent,train_within_50_percent,train_mae,train_rmse,train_rmsle,train_r2,validation_mape_percent,...,overfitting_flag,test_mape_percent,test_within_50_percent,test_mae,test_rmse,test_rmsle,test_r2,best_model_flag,error_vs_best_ratio,underperforming_flag
0,RandomForest (tuned),10205.572128,3628.447021,7.027162,99.696740,1524.358528,5748.579499,0.100540,0.949655,21.322660,...,True,20.270185,92.484848,3106.413558,7663.386268,0.278360,0.876338,True,1.000000,False
1,RandomForest,10205.572128,3628.447021,7.027162,99.696740,1524.358528,5748.579499,0.100540,0.949655,21.322660,...,True,20.270185,92.484848,3106.413558,7663.386268,0.278360,0.876338,True,1.000000,False
2,XGBoost,9702.833721,3114.578493,15.553063,96.702047,3064.369059,8079.883535,0.204147,0.900540,20.715625,...,False,19.792737,92.969697,3300.523180,8318.586218,0.269212,0.854289,False,1.085497,False
3,HistogramGradientBoostedTreesModel,10053.532935,3408.291713,10.994736,98.711145,2213.910413,6802.911506,0.150913,0.929494,21.008191,...,False,20.436698,93.333333,3309.792838,8320.032461,0.274914,0.854238,False,1.085686,False
4,GradientBoostedTreesModel,10484.749401,3265.457074,16.625682,96.019712,3342.677114,10264.970698,0.224581,0.839471,20.740916,...,False,20.247176,93.333333,3323.337918,8433.264747,0.278305,0.850244,False,1.100462,False
5,DecisionTree (tuned),12356.048857,3237.677416,19.604414,93.896892,4017.940810,10974.037105,0.259126,0.816528,27.619474,...,False,25.962200,90.424242,4354.397466,10108.062013,0.337899,0.784856,False,1.319007,True
6,LinearRegression,13845.594346,3334.636935,23.496234,89.992418,5002.563889,14174.740197,0.314827,0.693897,26.063161,...,False,24.351857,89.333333,4279.105670,10953.681023,0.319497,0.747353,False,1.429353,True
7,ElasticNet,13857.597717,3329.223066,23.506668,89.916603,5013.000474,14185.321312,0.314850,0.693440,26.042194,...,False,24.346407,89.333333,4289.954702,10967.191264,0.319343,0.746729,False,1.431116,True
8,Lasso,13865.841352,3326.830525,23.514383,89.954511,5020.019919,14192.389643,0.314893,0.693134,26.024351,...,False,24.335712,89.454545,4296.306767,10974.704179,0.319224,0.746382,False,1.432096,True
9,Ridge,13882.923451,3308.911637,23.560327,89.878696,5032.086653,14198.996973,0.315020,0.692848,26.058164,...,False,24.446328,89.454545,4317.058844,10993.828775,0.319679,0.745498,False,1.434591,True


Report: /Users/blaze/Desktop/python-ml-data-science/regression/car_price_prediction/reports/training_performance.html


Report: /Users/blaze/Desktop/python-ml-data-science/regression/car_price_prediction/reports/testing_evaluation.html


Report: /Users/blaze/Desktop/python-ml-data-science/regression/car_price_prediction/reports/model_comparison.html
